# Lab 5: Registro do Baseline - Naive RAG para Direito e Segurança Pública

## Objetivos da Aula

Este laboratório estabelece o **baseline Naive RAG** que será usado em comparações ao longo de TODAS as aulas subsequentes (aulas 3-12). Sem um baseline reproduzível, não é possível medir progresso em sistemas RAG.

### Três objetivos principais:
1. **Executar 5 queries de teste** cobrindo diferentes tipos de necessidade informacional jurídica
2. **Documentar respostas e artefatos** (chunks recuperados, tokens, tempos)
3. **Avaliar qualitativamente** nas 5 dimensões que importam em RAG jurídico

---

## Diagrama: O Baseline como Âncora

```
                         AULA 2: LAB 5
                    Baseline Naive RAG
                    (v0.0 do sistema)
                            |
                 ___________+___________
                 |         |           |
            AULA 3      AULA 5      AULA 7
          Reranking   Compressão  Query Exp.
                 |         |           |
                 |    AULA 8: Self-RAG |
                 |         |           |
                 |    AULA 10: HyDE    |
                 |         |           |
                 +_________+___________+
                           |
                    Resultado final:
                  Melhoria documentada
                 (baseline → v1.0 final)
```

---

## Referências ABNT

- LEWIS, P. et al. **Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks**. *NeurIPS*, 2020. Disponível em: https://arxiv.org/abs/2005.11401
- GAO, Y. et al. **Retrieval-Augmented Generation for Large Language Models: A Survey**. arXiv, 2023. Disponível em: https://arxiv.org/abs/2312.10997
- KHANDELWAL, U. et al. **Generalization through the Lens of Leave-One-Out Error**. *NeurIPS*, 2020.
- WANG, X.; WANG, W. **Is Retriever Merely a Plug-in for LLM?** *EMNLP*, 2023.

In [ ]:
# Célula 1: Instalação de dependências
# Bibliotecas necessárias para reproduzir o pipeline Naive RAG do LAB4 sobre o Ollama da Aula 1

import subprocess, sys

packages = [
    'langchain>=0.3',
    'langchain-community>=0.3',
    'langchain-core>=0.3',
    'langchain-ollama>=0.2',     # cliente nativo do Ollama (LLM + Embeddings)
    'langchain-openai>=0.2',     # alternativa via endpoint OpenAI-compat do Ollama
    'sentence-transformers>=3.0',# fallback HuggingFace para BGE-M3
    'faiss-cpu>=1.8',
    'opensearch-py>=2.7',
    'pandas>=2.2',
    'openpyxl>=3.1',
    'matplotlib>=3.9',
    'seaborn>=0.13',
    'jinja2>=3.1',
    'numpy>=1.26',
    'scikit-learn>=1.5',
    'tqdm>=4.66',
    'python-dotenv>=1.0',
    'requests>=2.32',
]

print('\n' + '='*60)
print('INSTALAÇÃO DE DEPENDÊNCIAS — PIPELINE RAG (Ollama da Aula 1)')
print('='*60 + '\n')

for package in packages:
    print(f'Instalando {package}...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print('\n' + '='*60)
print('VERIFICAÇÃO DE VERSÕES')
print('='*60 + '\n')

import langchain
from langchain_community.vectorstores import FAISS
import pandas as pd, numpy as np, matplotlib, seaborn as sns

print(f'✓ langchain: {langchain.__version__}')
print(f'✓ pandas: {pd.__version__}')
print(f'✓ numpy: {np.__version__}')
print(f'✓ matplotlib: {matplotlib.__version__}')
print(f'✓ seaborn: {sns.__version__}')
print(f'\n✓ Python: {sys.version.split()[0]}')
print('\n' + '='*60)
print('INSTALAÇÃO CONCLUÍDA')
print('='*60)

# Verificar Ollama da Aula 1
import os, requests
from dotenv import load_dotenv
load_dotenv()
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')
OLLAMA_LLM_MODEL = os.getenv('OLLAMA_LLM_MODEL', 'llama3.2:3b')
OLLAMA_EMBED_MODEL = os.getenv('OLLAMA_EMBED_MODEL', 'bge-m3')
try:
    r = requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=5)
    modelos = [m['name'] for m in r.json().get('models', [])]
    print(f'\n✓ Ollama OK em {OLLAMA_BASE_URL} | modelos: {modelos}')
except Exception as e:
    print(f'\n❌ Ollama não responde em {OLLAMA_BASE_URL} — inicie com `ollama serve` antes de continuar.')
    print(f'   Erro: {e}')


## Conceitos: O que é um Bom Baseline RAG?

Um baseline deve ter **4 propriedades críticas**:

### 1. **Reproduzível**
- Mesmos dados de entrada → mesmos resultados
- Não depende de API externa não-determinística
- Seeds fixadas para embeddings e geração

### 2. **Documentado**
- Versão exata de cada componente (modelo, LLM, índice)
- Hiperparâmetros explícitos (k=5, chunk_size=512, etc)
- Todos os passos registrados

### 3. **Auditável**
- Cada decisão é rastreável até sua fonte
- Chunks recuperados visíveis e avaliáveis
- Scores de similaridade capturados

### 4. **Comparável**
- Mesmas queries em todas as versões
- Mesmas métricas avaliadas
- Formato exportável (CSV, JSON) para análise cruzada

### Estrutura de um Benchmark Manual

```
Query → Embedding → Retrieval (k=5) → Ranking → LLM → Resposta
         ↓          ↓                           ↓
      Tempo      Chunks                    Tokens
      (ms)      Recuperados              Gerados
                Scores
```

Cada ponto medido permite diagnóstico de gargalos.

In [ ]:
# Célula 2: Inicializar pipeline RAG sobre o Ollama da Aula 1
# - Embeddings:  bge-m3   via Ollama  (fallback: HuggingFace local)
# - LLM:         llama3.2:3b via Ollama  (fallback: modo simulado para defesa offline)
# - Vector store: FAISS in-memory (espelha o que o LAB4 já demonstrou com OpenSearch)

import os, time, json, requests, warnings
from typing import Any
from datetime import datetime
from langchain_community.vectorstores import FAISS
from langchain.schema import Document
from langchain.prompts import PromptTemplate
from dotenv import load_dotenv
warnings.filterwarnings('ignore')
load_dotenv()

OLLAMA_BASE_URL    = os.getenv('OLLAMA_BASE_URL',    'http://localhost:11434')
OLLAMA_LLM_MODEL   = os.getenv('OLLAMA_LLM_MODEL',   'llama3.2:3b')
OLLAMA_EMBED_MODEL = os.getenv('OLLAMA_EMBED_MODEL', 'bge-m3')

print('\n' + '='*60)
print('INICIALIZAÇÃO DO PIPELINE RAG (BASELINE — AULA 2)')
print('='*60 + '\n')

# ========== 1. EMBEDDINGS: BGE-M3 via Ollama (fallback HuggingFace) ==========
print(f'[1/4] Carregando embeddings BGE-M3...')
start_embed = time.time()
embedding_model = None

try:
    from langchain_ollama import OllamaEmbeddings
    requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=3).raise_for_status()
    embedding_model = OllamaEmbeddings(model=OLLAMA_EMBED_MODEL, base_url=OLLAMA_BASE_URL)
    dim_real = len(embedding_model.embed_query('teste'))
    embed_backend = f'Ollama({OLLAMA_EMBED_MODEL})'
    print(f'   ✓ {embed_backend} | dim={dim_real}')
except Exception as e:
    print(f'   ⚠️  Ollama embeddings indisponível ({e}). Caindo para HuggingFace.')
    from langchain_community.embeddings import HuggingFaceEmbeddings
    embedding_model = HuggingFaceEmbeddings(
        model_name='BAAI/bge-m3',
        model_kwargs={'trust_remote_code': True, 'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True},
    )
    dim_real = len(embedding_model.embed_query('teste'))
    embed_backend = 'HuggingFace(BAAI/bge-m3)'
    print(f'   ✓ {embed_backend} | dim={dim_real}')

embed_time = time.time() - start_embed
print(f'   ⏱️  setup embeddings: {embed_time:.2f}s\n')

# ========== 2. CORPUS MÍNIMO JURÍDICO ==========
corpus_docs = [
    Document(
        page_content='Lei 11.343/2006 (Lei de Drogas) - Art. 33: Importar, exportar, remeter, preparar, produzir, fabricar, adquirir, vender, distribuir, entregar, possuir, guardar ou fornecer, ainda que gratuitamente, sem autorização ou em desacordo com determinação legal ou regulamentar, substância ou produto capaz de causar dependência física ou psíquica. Pena: reclusão de 5 a 15 anos e multa de 500 a 1.500 UFIRs.',
        metadata={'source': 'Lei 11.343/2006', 'tipo': 'legislacao', 'artigo': '33'}
    ),
    Document(
        page_content='STJ - Entendimento consolidado: A fundamentação da prisão preventiva baseada exclusivamente na gravidade abstrata do delito é insuficiente. Exige-se demonstração concreta de risco ao direito processual (HC 470.819/RJ, DJ 02/12/2019).',
        metadata={'source': 'STJ', 'tipo': 'jurisprudencia', 'tribunal': 'STJ'}
    ),
    Document(
        page_content='Relatório de Inteligência 1º Trimestre 2025: Identificados 47 pontos críticos de tráfico. Recomendações: (1) Aumento de patrulhamento noturno em 30%; (2) Integração com fiscalização municipal; (3) Análise de redes de distribuição via inteligência artificial.',
        metadata={'source': 'Relatorio_Inteligencia_Q1_2025', 'tipo': 'operacional', 'periodo': '2025-Q1'}
    ),
    Document(
        page_content='Estatísticas de Segurança Pública - Tráfico de Drogas: 1º trimestre 2024: 523 ocorrências; 1º trimestre 2025: 596 ocorrências. Variação: +13,8%. Apreensões: 156 kg em 2024 vs 189 kg em 2025 (+21,2%).',
        metadata={'source': 'Dados_Seguranca_Publica', 'tipo': 'estatistico', 'periodo': '2024-2025'}
    ),
    Document(
        page_content='Análise Jurisprudencial: A quantidade de 2g de cocaína é considerada para uso pessoal (entendimento jurisprudencial). Acima de 5g presume-se tráfico. Entre 2-5g cabe análise contextual (local, comportamento, padrão de vida, antecedentes).',
        metadata={'source': 'Analise_Jurisprudencial_Drogas', 'tipo': 'analise', 'tema': 'trafego_vs_uso_pessoal'}
    ),
]

print('[2/4] Criando índice FAISS com corpus jurídico...')
start_faiss = time.time()
vectorstore = FAISS.from_documents(corpus_docs, embedding_model)
faiss_time = time.time() - start_faiss
print(f'   ✓ FAISS criado com {len(corpus_docs)} documentos em {faiss_time:.2f}s\n')

# ========== 3. LLM: Ollama (fallback simulado para defesa offline) ==========
print('[3/4] Configurando LLM via Ollama...')
llm = None
llm_mode = None
try:
    from langchain_ollama import ChatOllama
    requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=3).raise_for_status()
    llm = ChatOllama(
        model=OLLAMA_LLM_MODEL,
        base_url=OLLAMA_BASE_URL,
        temperature=0,        # determinístico para baseline reprodutível
        num_predict=512,
    )
    # Smoke test
    _ = llm.invoke('OK em uma palavra:')
    llm_mode = f'Ollama({OLLAMA_LLM_MODEL}) em {OLLAMA_BASE_URL}'
    print(f'   ✓ {llm_mode}')
except Exception as e:
    print(f'   ⚠️  Ollama LLM indisponível ({e}). Usando modo simulado determinístico.')
    class SimulatedLLM:
        def __init__(self):
            self.model = 'simulated-deterministic'
            self.temperature = 0
        def invoke(self, prompt: str) -> Any:
            return type('obj', (object,), {'content': 'Resposta simulada (modo offline): ' + str(prompt)[:100] + '...'})()
    llm = SimulatedLLM()
    llm_mode = 'Simulado (offline) — APENAS para demonstração sem rede'

print(f'   ✓ LLM mode: {llm_mode}\n')

# ========== 4. PROMPT TEMPLATE JURÍDICO (mesmo do LAB4) ==========
print('[4/4] Configurando template de prompt jurídico...\n')

template_juridico = """Você é um assistente jurídico especializado em direito penal e segurança pública.

### INSTRUÇÃO CRÍTICA: SEMPRE CITE AS FONTES

Você receberá um contexto (documentos) e uma pergunta. Responda APENAS com informações que estão no contexto.

**REGRA 1**: Se a informação não está no contexto, responda "Não há informação suficiente no contexto para responder."
**REGRA 2**: Cada afirmação DEVE ter citação de fonte no formato [Fonte: Nome da Lei/Documento]
**REGRA 3**: Se tiver múltiplas interpretações, indique qual aparece no contexto

### CONTEXTO (Documentos Recuperados)

{context}

### PERGUNTA

{question}

### RESPOSTA

Responda em português, de forma jurídica, com todas as fontes citadas."""

prompt_template = PromptTemplate(
    input_variables=['context', 'question'],
    template=template_juridico,
)

print('='*60)
print('PIPELINE RAG INICIALIZADO COM SUCESSO')
print('='*60)
print(f'\nConfigurações do baseline:')
print(f'  • Embeddings: {embed_backend} (dim={dim_real})')
print(f'  • Índice: FAISS ({len(corpus_docs)} docs)')
print(f'  • LLM: {llm_mode}')
print(f'  • K (chunks recuperados): 5')
print(f'  • Temperatura: 0 (determinístico)')
print(f'  • Tempo total de setup: {embed_time + faiss_time:.2f}s')
print('='*60)


## As 5 Queries de Baseline

As queries foram selecionadas para cobrir **5 tipos diferentes de necessidade informacional jurídica**, cada uma desafiando aspectos diferentes do RAG:

| ID | Categoria | Desafio | Esperado do Naive RAG |
|---|---|---|---|
| Q1 | **Factual** | Precisa de números e dados atualizados | Bom (dados estão no corpus) |
| Q2 | **Legal** | Extração de artigo/pena específica | Excelente (texto é literal) |
| Q3 | **Jurisprudencial** | Síntese de posição consolidada | Médio (requer inferência) |
| Q4 | **Operacional** | Procedimentos e recomendações | Médio-Bom (estruturado no doc) |
| Q5 | **Comparativa/Analítica** | Relação entre conceitos | Fraco (requer análise cruzada) |

### Estratégia de Construção
- **Q1-Q2**: Queries que o Naive RAG deve responder bem (validação positiva)
- **Q3-Q4**: Queries com dificuldade média (mostram limites)
- **Q5**: Query complexa que expõe fraqueza de análise cruzada

In [ ]:
# Célula 3: Definir as 5 Queries de Baseline
# Estrutura: id, categoria, texto, resposta esperada, documentos relevantes

queries_baseline = [
    {
        "id": "Q1",
        "categoria": "Factual",
        "texto": "Quantas ocorrências de tráfico de drogas foram registradas no 1º trimestre de 2025 e qual foi a variação percentual em relação ao ano anterior?",
        "resposta_esperada": "1º trim 2025: 596 ocorrências (variação +13,8% vs 523 em 2024)",
        "documentos_relevantes": ["Dados_Seguranca_Publica"]
    },
    {
        "id": "Q2",
        "categoria": "Legal",
        "texto": "Qual é a pena prevista no artigo 33 da Lei 11.343/2006 para o crime de tráfico de drogas?",
        "resposta_esperada": "Reclusão de 5 a 15 anos e multa de 500 a 1.500 UFIRs",
        "documentos_relevantes": ["Lei 11.343/2006"]
    },
    {
        "id": "Q3",
        "categoria": "Jurisprudencial",
        "texto": "Qual é o entendimento consolidado do STJ sobre a fundamentação da prisão preventiva baseada exclusivamente na gravidade abstrata do delito?",
        "resposta_esperada": "Insuficiente; exige demonstração concreta de risco ao direito processual (HC 470.819/RJ)",
        "documentos_relevantes": ["STJ"]
    },
    {
        "id": "Q4",
        "categoria": "Operacional",
        "texto": "Quais são as recomendações operacionais do relatório de inteligência para combate ao tráfico de drogas nos pontos críticos identificados?",
        "resposta_esperada": "(1) Aumento patrulhamento noturno 30%, (2) Integração com fiscalização municipal, (3) Análise de redes via IA",
        "documentos_relevantes": ["Relatorio_Inteligencia_Q1_2025"]
    },
    {
        "id": "Q5",
        "categoria": "Analítica",
        "texto": "Como a quantidade de droga apreendida se relaciona com a distinção entre tráfico e uso pessoal segundo a legislação e a jurisprudência analisada?",
        "resposta_esperada": "Até 2g: uso pessoal; 5g+: presume tráfico; 2-5g: análise contextual (local, padrão vida, antecedentes)",
        "documentos_relevantes": ["Analise_Jurisprudencial_Drogas", "Lei 11.343/2006"]
    }
]

# ========== APRESENTAÇÃO DAS QUERIES ==========
print("\n" + "="*70)
print("QUERIES DE BASELINE - DEFINIÇÃO FORMAL")
print("="*70 + "\n")

import pandas as pd

# Dataframe para visualização
df_queries = pd.DataFrame([
    {
        "ID": q["id"],
        "Categoria": q["categoria"],
        "Query": q["texto"][:60] + "...",
        "Documentos Esperados": ", ".join(q["documentos_relevantes"])
    }
    for q in queries_baseline
])

print(df_queries.to_string(index=False))
print("\n" + "="*70 + "\n")

# Exibir cada query em detalhes
for i, q in enumerate(queries_baseline, 1):
    print(f"\n┌─ QUERY {q['id']}: {q['categoria'].upper()}")
    print(f"├─ Texto: {q['texto']}")
    print(f"├─ Resposta Esperada: {q['resposta_esperada']}")
    print(f"└─ Documentos Relevantes: {', '.join(q['documentos_relevantes'])}")

print("\n" + "="*70)
print(f"TOTAL: {len(queries_baseline)} queries definidas para baseline")
print("="*70)

## Execução e Coleta de Respostas

Nesta seção, executamos cada query através do pipeline RAG completo:

```
Query → [EMBEDDING] → [RETRIEVAL] → [LLM] → Response
         ↓ tempo      ↓ chunks      ↓ tempo
         medido       medido        medido
```

Coletamos:
- Chunks recuperados e seus scores
- Tempo de embedding
- Tempo de LLM
- Resposta completa

In [ ]:
# Célula 4: Executar as 5 queries e coletar respostas
# Função: executar_query_completa() com medição de tempo e artefatos

from tqdm import tqdm

def executar_query_completa(query_info: Dict, vectorstore, llm, prompt_template, embedding_model) -> Dict[str, Any]:
    """
    Executa uma query completa através do pipeline RAG e coleta todos os artefatos.
    
    Args:
        query_info: Dicionário com id, texto, categoria, etc.
        vectorstore: Índice FAISS
        llm: Modelo de linguagem
        prompt_template: Template de prompt jurídico
        embedding_model: Modelo de embeddings
    
    Returns:
        Dicionário com todos os resultados e métricas
    """
    
    result = {
        "id": query_info["id"],
        "categoria": query_info["categoria"],
        "query_texto": query_info["texto"],
        "resposta_esperada": query_info["resposta_esperada"],
        "timestamp": datetime.now().isoformat()
    }
    
    # ===== PASSO 1: EMBEDDING DA QUERY =====
    inicio_embedding = time.time()
    query_embedding = embedding_model.embed_query(query_info["texto"])
    tempo_embedding = time.time() - inicio_embedding
    result["tempo_embedding_ms"] = round(tempo_embedding * 1000, 2)
    result["tamanho_embedding"] = len(query_embedding)
    
    # ===== PASSO 2: RETRIEVAL (busca no FAISS) =====
    inicio_retrieval = time.time()
    docs_recuperados = vectorstore.similarity_search_with_scores(query_info["texto"], k=5)
    tempo_retrieval = time.time() - inicio_retrieval
    result["tempo_retrieval_ms"] = round(tempo_retrieval * 1000, 2)
    result["num_chunks_recuperados"] = len(docs_recuperados)
    
    # Armazenar chunks com metadados e scores
    chunks_info = []
    contexto_para_llm = ""
    for i, (doc, score) in enumerate(docs_recuperados, 1):
        chunk_entry = {
            "rank": i,
            "score_similaridade": round(float(score), 4),
            "conteudo": doc.page_content[:200] + "..." if len(doc.page_content) > 200 else doc.page_content,
            "fonte": doc.metadata.get("source", "desconhecida")
        }
        chunks_info.append(chunk_entry)
        contexto_para_llm += f"[{i}] (Fonte: {doc.metadata.get('source', 'N/A')})\n{doc.page_content}\n\n"
    
    result["chunks"] = chunks_info
    result["contexto_recuperado"] = contexto_para_llm
    
    # ===== PASSO 3: GERAÇÃO COM LLM =====
    inicio_llm = time.time()
    prompt_final = prompt_template.format(context=contexto_para_llm, question=query_info["texto"])
    
    # Chamar LLM
    if isinstance(llm, SimulatedLLM):
        # Fallback: resposta simulada determinística
        resposta_llm = f"Resposta simulada para {query_info['id']}: {query_info['resposta_esperada']}"
        tokens_gerados = len(resposta_llm.split())
    else:
        # Real OpenAI API
        try:
            msg = llm.invoke(prompt_final)
            resposta_llm = msg.content if hasattr(msg, 'content') else str(msg)
            tokens_gerados = len(resposta_llm.split())
        except:
            resposta_llm = "[Erro ao chamar API]"
            tokens_gerados = 0
    
    tempo_llm = time.time() - inicio_llm
    result["tempo_llm_ms"] = round(tempo_llm * 1000, 2)
    result["resposta"] = resposta_llm
    result["tokens_estimados"] = tokens_gerados
    result["tempo_total_ms"] = round((tempo_embedding + tempo_retrieval + tempo_llm) * 1000, 2)
    
    return result

print("\n" + "="*70)
print("EXECUTANDO QUERIES DE BASELINE")
print("="*70 + "\n")

# Executar todas as queries
resultados_queries = []
for query_info in tqdm(queries_baseline, desc="Processando queries", unit="query"):
    resultado = executar_query_completa(
        query_info,
        vectorstore,
        llm,
        prompt_template,
        embedding_model
    )
    resultados_queries.append(resultado)
    
    # Exibir resultado da query
    print(f"\n{'='*70}")
    print(f"[{resultado['id']}] {resultado['categoria'].upper()}")
    print(f"{'='*70}")
    print(f"\nQuery: {resultado['query_texto']}")
    print(f"\nResposta Esperada: {resultado['resposta_esperada']}")
    print(f"\nResposta Gerada:\n{resultado['resposta'][:300]}...\n")
    print(f"Métricas:")
    print(f"  • Tempo embedding: {resultado['tempo_embedding_ms']:.2f}ms")
    print(f"  • Tempo retrieval: {resultado['tempo_retrieval_ms']:.2f}ms")
    print(f"  • Tempo LLM: {resultado['tempo_llm_ms']:.2f}ms")
    print(f"  • Tempo total: {resultado['tempo_total_ms']:.2f}ms")
    print(f"  • Chunks recuperados: {resultado['num_chunks_recuperados']}")
    print(f"  • Tokens estimados: {resultado['tokens_estimados']}")

print(f"\n{'='*70}")
print(f"EXECUÇÃO CONCLUÍDA: {len(resultados_queries)} queries processadas")
print(f"{'='*70}")

## Avaliação Qualitativa Manual

Agora avaliamos cada resposta nas **5 dimensões críticas para RAG jurídico**:

### Dimensão 1: Relevância do Contexto Recuperado (retrieval_relevance)
- **5**: Todos os chunks são diretamente relevantes
- **3**: Alguns chunks relevantes, alguns periféricos
- **1**: Maioria não é relevante

### Dimensão 2: Completude da Resposta (answer_completeness)
- **5**: Contém TODAS as informações necessárias
- **3**: Parcial — informação principal presente mas faltam detalhes
- **1**: Incompleta ou superficial

### Dimensão 3: Fidelidade ao Contexto (faithfulness)
- **5**: Toda informação é rastreável aos docs
- **3**: Maioria rastreável, 1-2 pontos vagos
- **1**: Contém informações inventadas

### Dimensão 4: Qualidade das Citações (citation_quality)
- **5**: Cada afirmação tem fonte citada
- **3**: Algumas fontes citadas
- **1**: Sem citações ou incorretas

### Dimensão 5: Utilidade Jurídica (legal_utility)
- **5**: Profissional pode usar diretamente
- **3**: Útil como ponto de partida, requer verificação
- **1**: Não seria utilizado sem revisão extensiva

In [ ]:
# Célula 5: Framework de avaliação qualitativa com scoring pré-preenchido
# Demonstração com scores realistas e espaços para o aluno completar

# Definição das dimensões de avaliação
dimensoes_avaliacao = [
    {"id": "retrieval_relevance", "nome": "Relevância do Contexto Recuperado", "peso": 1.0},
    {"id": "answer_completeness", "nome": "Completude da Resposta", "peso": 1.0},
    {"id": "faithfulness", "nome": "Fidelidade ao Contexto / Anti-Alucinação", "peso": 1.2},
    {"id": "citation_quality", "nome": "Qualidade das Citações", "peso": 1.1},
    {"id": "legal_utility", "nome": "Utilidade Jurídica", "peso": 1.0}
]

# ===== SCORES PRÉ-PREENCHIDOS PARA DEMONSTRAÇÃO =====
# Baseados em perfil típico do Naive RAG
scores_demonstracao = {
    "Q1": {  # Query factual: deve ir bem
        "retrieval_relevance": 5,
        "answer_completeness": 5,
        "faithfulness": 5,
        "citation_quality": 4,
        "legal_utility": 5
    },
    "Q2": {  # Query legal simples: vai excelente
        "retrieval_relevance": 5,
        "answer_completeness": 5,
        "faithfulness": 5,
        "citation_quality": 5,
        "legal_utility": 5
    },
    "Q3": {  # Query jurisprudencial: médio
        "retrieval_relevance": 4,
        "answer_completeness": 3,
        "faithfulness": 4,
        "citation_quality": 3,
        "legal_utility": 3
    },
    "Q4": {  # Query operacional estruturada: bom
        "retrieval_relevance": 5,
        "answer_completeness": 4,
        "faithfulness": 4,
        "citation_quality": 4,
        "legal_utility": 4
    },
    "Q5": {  # Query analítica complexa: fraco
        "retrieval_relevance": 3,
        "answer_completeness": 2,
        "faithfulness": 3,
        "citation_quality": 2,
        "legal_utility": 2
    }
}

# Construir dataframe de avaliação
df_avaliacao = pd.DataFrame()

for query_id, scores_query in scores_demonstracao.items():
    linha = {"Query": query_id}
    for dim in dimensoes_avaliacao:
        linha[dim["nome"]] = scores_query[dim["id"]]
    # Calcular score médio da query
    scores_list = list(scores_query.values())
    linha["Score Médio"] = round(sum(scores_list) / len(scores_list), 2)
    df_avaliacao = pd.concat([df_avaliacao, pd.DataFrame([linha])], ignore_index=True)

print("\n" + "="*120)
print("MATRIZ DE AVALIAÇÃO QUALITATIVA - NAIVE RAG BASELINE")
print("Escala: 1 (fraco) a 5 (excelente)")
print("="*120 + "\n")

print(df_avaliacao.to_string(index=False))

print("\n" + "="*120)
print("LEGENDA E INSTRUÇÕES")
print("="*120)
print("""
Cada dimensão deve ser preenchida manualmente após revisar a resposta:

1. RELEVÂNCIA DO CONTEXTO RECUPERADO (Escala 1-5)
   5 = Todos os chunks são diretamente relevantes
   3 = Alguns chunks relevantes, alguns periféricos  
   1 = Maioria dos chunks não é relevante
   Critério: O FAISS trouxe os documentos corretos?

2. COMPLETUDE DA RESPOSTA (Escala 1-5)
   5 = Resposta contém TODAS as informações necessárias
   3 = Parcial — informação principal presente mas faltam detalhes
   1 = Incompleta ou superficial
   Critério: A resposta responde a TODOS os aspectos da pergunta?

3. FIDELIDADE AO CONTEXTO (Escala 1-5)
   5 = Toda informação é rastreável aos documentos recuperados
   3 = Maioria rastreável, 1-2 pontos vagos
   1 = Contém informações inventadas (alucinação)
   Critério: É possível verificar cada afirmação no contexto?

4. QUALIDADE DAS CITAÇÕES (Escala 1-5)
   5 = Cada afirmação tem fonte citada no formato correto [Fonte: ...]
   3 = Algumas fontes citadas, mas não todas
   1 = Sem citações ou citações incorretas
   Critério: Posso rastrear a origem de cada informação?

5. UTILIDADE JURÍDICA (Escala 1-5)
   5 = Profissional jurídico pode usar a resposta diretamente
   3 = Útil como ponto de partida, mas requer verificação adicional
   1 = Não seria utilizado sem revisão extensiva
   Critério: Um advogado/promotor usaria essa resposta em um parecer?

PREENCHIDO COM VALORES DE DEMONSTRAÇÃO: 'Score Médio' mostra a média das 5 dimensões.
ALUNO: Revise cada score após ler as respostas geradas e ajuste conforme necessário.
""")
print("="*120)

## Calculando as Métricas Agregadas do Baseline

In [ ]:
# Célula 7: Visualizações do baseline com 4 gráficos
# Radar chart, heatmap, e barplots de performance

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from math import pi

# Configurar estilo
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 12)
plt.rcParams['font.size'] = 10

fig = plt.figure(figsize=(16, 12))

# ===== GRÁFICO 1: RADAR CHART (Spider) =====
ax1 = plt.subplot(2, 2, 1, projection='polar')

categories = [dim[:20] + '...' if len(dim) > 20 else dim for dim in dimensoes_nomes]
values = [scores_dimensao[dim] for dim in dimensoes_nomes]
values += values[:1]  # Fechar o polígono

angles = [n / float(len(categories)) * 2 * pi for n in range(len(categories))]
angles += angles[:1]

ax1.plot(angles, values, 'o-', linewidth=2, color='#2E86AB', label='Naive RAG')
ax1.fill(angles, values, alpha=0.25, color='#2E86AB')
ax1.set_xticks(angles[:-1])
ax1.set_xticklabels(categories, size=8)
ax1.set_ylim(0, 5)
ax1.set_yticks([1, 2, 3, 4, 5])
ax1.set_title('Perfil de Desempenho: Naive RAG', size=12, weight='bold', pad=20)
ax1.grid(True)
ax1.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

# ===== GRÁFICO 2: HEATMAP Query × Dimensão =====
ax2 = plt.subplot(2, 2, 2)

heatmap_data = df_avaliacao[['Query'] + dimensoes_nomes].set_index('Query')
sns.heatmap(heatmap_data, annot=True, fmt='g', cmap='RdYlGn', vmin=1, vmax=5,
            ax=ax2, cbar_kws={'label': 'Score'}, linewidths=0.5)
ax2.set_title('Matriz: Query × Dimensão de Avaliação', size=12, weight='bold')
ax2.set_xlabel('Dimensão de Qualidade', fontweight='bold')
ax2.set_ylabel('Query de Teste', fontweight='bold')

# ===== GRÁFICO 3: TEMPOS DE RESPOSTA =====
ax3 = plt.subplot(2, 2, 3)

queries_ids = [r['id'] for r in resultados_queries]
tempos_emb = [r['tempo_embedding_ms'] for r in resultados_queries]
tempos_ret = [r['tempo_retrieval_ms'] for r in resultados_queries]
tempos_llm_list = [r['tempo_llm_ms'] for r in resultados_queries]

x = np.arange(len(queries_ids))
width = 0.25

ax3.bar(x - width, tempos_emb, width, label='Embedding', color='#A23B72')
ax3.bar(x, tempos_ret, width, label='Retrieval', color='#F18F01')
ax3.bar(x + width, tempos_llm_list, width, label='LLM', color='#2E86AB')

ax3.set_ylabel('Tempo (ms)', fontweight='bold')
ax3.set_xlabel('Query', fontweight='bold')
ax3.set_title('Decomposição de Tempo por Etapa do Pipeline', size=12, weight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels(queries_ids)
ax3.legend()
ax3.grid(axis='y', alpha=0.3)

# ===== GRÁFICO 4: SCORES POR QUERY =====
ax4 = plt.subplot(2, 2, 4)

scores_por_query = df_avaliacao['Score Médio'].tolist()
colors = ['#2E86AB' if s >= 4 else '#F18F01' if s >= 3 else '#A23B72' for s in scores_por_query]

bars = ax4.barh(queries_ids, scores_por_query, color=colors, edgecolor='black', linewidth=1.5)
ax4.set_xlabel('Score Médio (1-5)', fontweight='bold')
ax4.set_title('Score Geral por Query (Todas as Dimensões)', size=12, weight='bold')
ax4.set_xlim(0, 5)
ax4.grid(axis='x', alpha=0.3)

# Adicionar valores nos barras
for i, (bar, score) in enumerate(zip(bars, scores_por_query)):
    ax4.text(score + 0.1, bar.get_y() + bar.get_height()/2, f'{score:.2f}',
             va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/baseline_visualizacoes.png', dpi=150, bbox_inches='tight')
print("\n✓ Gráficos salvos em /tmp/baseline_visualizacoes.png")
plt.show()

print("\n" + "="*70)
print("VISUALIZAÇÕES DO BASELINE GERADAS")
print("="*70)

## Exportando o Baseline em 3 Formatos

In [ ]:
# Célula 8: Exportar baseline em CSV, Excel e JSON
import json
from datetime import datetime

print("\n" + "="*70)
print("EXPORTANDO BASELINE EM 3 FORMATOS")
print("="*70 + "\n")

# ===== FORMATO 1: CSV =====
df_export_csv = pd.DataFrame([
    {
        'query_id': row['Query'],
        'categoria': next(r['categoria'] for r in resultados_queries if r['id'] == row['Query']),
        'score_medio': row['Score Médio'],
        'retrieval_relevance': row['Relevância do Contexto Recuperado'],
        'answer_completeness': row['Completude da Resposta'],
        'faithfulness': row['Fidelidade ao Contexto / Anti-Alucinação'],
        'citation_quality': row['Qualidade das Citações'],
        'legal_utility': row['Utilidade Jurídica']
    }
    for _, row in df_avaliacao.iterrows()
])

csv_path = '/tmp/baseline_naive_rag_aula2.csv'
df_export_csv.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f"✓ CSV exportado: baseline_naive_rag_aula2.csv")
print(f"  Linhas: {len(df_export_csv)}\n")

# ===== FORMATO 2: EXCEL =====
with pd.ExcelWriter('/tmp/baseline_aula2.xlsx', engine='openpyxl') as writer:
    df_respostas = pd.DataFrame([
        {
            'Query ID': r['id'],
            'Categoria': r['categoria'],
            'Tempo Total (ms)': r['tempo_total_ms']
        }
        for r in resultados_queries
    ])
    df_respostas.to_excel(writer, sheet_name='Respostas', index=False)
    df_avaliacao.to_excel(writer, sheet_name='Scores', index=False)

print(f"✓ Excel exportado: baseline_aula2.xlsx\n")

# ===== FORMATO 3: JSON =====
baseline_json = {
    'metadata': {
        'aula': 'Aula 2 - Lab 5',
        'data_criacao': datetime.now().isoformat(),
        'versao': '0.0'
    },
    'metricas': {
        'score_geral': round(score_geral, 2),
        'num_queries': len(resultados_queries)
    }
}

json_path = '/tmp/baseline_aula2.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(baseline_json, f, indent=2, ensure_ascii=False)

print(f"✓ JSON exportado: baseline_aula2.json")
print("\n" + "="*70)
print("EXPORTAÇÃO CONCLUÍDA")
print("="*70)

## Análise de Fraquezas e Roadmap

In [ ]:
# Célula 9: Análise de fraquezas e mapeamento para aulas futuras

print("\n" + "="*80)
print("ANÁLISE DE FRAQUEZAS: ROADMAP DE MELHORIAS (AULA 3-12)")
print("="*80 + "\n")

dimensoes_ranking = [
    ("Relevância do Contexto Recuperado", scores_dimensao.get("Relevância do Contexto Recuperado", 3)),
    ("Completude da Resposta", scores_dimensao.get("Completude da Resposta", 3)),
    ("Fidelidade ao Contexto / Anti-Alucinação", scores_dimensao.get("Fidelidade ao Contexto / Anti-Alucinação", 3)),
    ("Qualidade das Citações", scores_dimensao.get("Qualidade das Citações", 3)),
    ("Utilidade Jurídica", scores_dimensao.get("Utilidade Jurídica", 3))
]

sorted_dims_weak = sorted(dimensoes_ranking, key=lambda x: x[1])

print("TOP 3 DIMENSÕES MAIS FRACAS:\\n")
for rank, (nome, score) in enumerate(sorted_dims_weak[:3], 1):
    print(f"#{rank}. {nome}: {score:.2f}/5.0")

print(f"\nSCORE GERAL DO BASELINE: {score_geral:.2f}/5.0")
print(f"META (após todas as aulas): 4.0+/5.0")
print(f"MELHORIA ESPERADA: +{4.0 - score_geral:.2f} pontos")

print("\n" + "="*80)
print("FIM DO NOTEBOOK - BASELINE PRONTO PARA AULA 3")
print("="*80)